# TFM Tenerife — Embeddings del corpus para el RAG (Fase 2)

Calcula el vector de 768 dimensiones de cada uno de los ~88.000 fragmentos de `gold.nlp_chunks`. Se hace aquí y no en local porque en CPU son horas y en una T4 gratuita son minutos.

**Activa la GPU**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU**.

**Pasos**:
1. Ejecuta antes `python analytics/rag/export_chunks.py` en local.
2. Sube `chunks_para_embeddings.csv` a tu Google Drive (raíz "Mi unidad").
3. Ejecuta todas las celdas (Entorno de ejecución → Ejecutar todo). La celda 2 pedirá autorizar Drive.
4. Descarga `embeddings.npz` y cárgalo con `python analytics/rag/import_embeddings.py embeddings.npz`.

El modelo es `paraphrase-multilingual-mpnet-base-v2`, **el mismo que usa BERTopic** en este proyecto: así el RAG y el modelado de tópicos comparten espacio vectorial.

In [ ]:
!pip install -q sentence-transformers

import numpy as np
import pandas as pd
import torch

print('GPU disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('AVISO: sin GPU esto tardara horas. Activa T4 en Entorno de ejecucion.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Se lee desde Drive y no con files.upload() porque el widget de subida
# corrompe archivos grandes con conexiones lentas (ya paso con el corpus de
# BERTopic). Ajusta la ruta si lo guardaste en una subcarpeta.
RUTA_CSV = '/content/drive/MyDrive/chunks_para_embeddings.csv'
print(f'Leyendo desde: {RUTA_CSV}')

In [ ]:
df = pd.read_csv(RUTA_CSV)
print(f'{len(df)} fragmentos cargados.')

# El orden importa: los embeddings se guardan alineados con esta lista de ids.
chunk_ids = df['chunk_id'].to_numpy(dtype=np.int64)
textos = df['text'].astype(str).tolist()
print('Ejemplo:', textos[0][:150])

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO = 'paraphrase-multilingual-mpnet-base-v2'  # el mismo de BERTopic
modelo = SentenceTransformer(MODELO, device='cuda' if torch.cuda.is_available() else 'cpu')

# normalize_embeddings=True deja todos los vectores con norma 1. Asi la
# similitud coseno equivale al producto escalar y el operador <=> de pgvector
# se comporta de forma consistente.
embeddings = modelo.encode(
    textos,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print('Forma resultante:', embeddings.shape)

In [ ]:
from google.colab import files

# float16 en vez de float32: reduce la descarga a la mitad (~135 MB en vez de
# ~270 MB) y la perdida de precision es irrelevante para ordenar por cercania,
# que es lo unico que se hace con estos vectores.
np.savez_compressed(
    'embeddings.npz',
    chunk_ids=chunk_ids,
    embeddings=embeddings.astype(np.float16),
    modelo=np.array([MODELO]),
)
print('Guardado embeddings.npz')
files.download('embeddings.npz')